# Suite EXP — Expressions

An expression is an `Expressions.Framework.Expression`, a reference object whose `root` is a term: a value of
`Expressions.Framework.Term`, the union of the dialects' `Term`s: Patterns' `Term`, `Expressions.Patterns.Expressions.Term`, lists
the framework's kinds `literal`, `call`, `variable` and `let`, the container constructors `list` and `mapping`, the
kinds of invalid syntax `error` and `missing`, and its own `quantifier`, and its install adds it to `Term` by schema
extension. A term carries no domain.
`Framework.install` and `Patterns.install` define them in a store; terms are built there,
in a transaction, through mbse-schemas' builders (or Patterns' `Writer`s, which write through them), and read through its
readers, so they serialize, validate and compare like any other value. `Evaluators` computes them (suite EVL).
A term is validated within its holder: `Patterns.define`, run after the installs, writes Patterns' operations, which
validation reads from the holder's store, so the suite's `term` gives an expression's root.

In [ ]:
import { Framework } from "@mbse/expressions";
import { Simplifiers as V, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import { Domains as FD, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Comparison as C, Errors, JSON, Plain, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, form_equal, map, raises, same } from "./support.js";

const { ValueError } = Errors;

const INSTALLS = [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, Patterns.install];
const store: any = new Builders.Store();
for (const install of INSTALLS) store.transaction(install); // the framework, the standard dialects, then Patterns: each its own transaction
store.transaction(Patterns.define); // Patterns' operations, which validation reads
const Term: any = store.registered("Expressions.Framework.Term");
const Expression: any = store.registered("Expressions.Framework.Expression");

/** The term `spec` writes, the root of a holder built in a transaction: validation reads its holder's store. */
function term(spec: unknown): any {
  return store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => e.root(E.DIALECT.spec(spec)))));
}

const lit = term(E.literal(3n));
const op = term(E.operation("f", 1n, "x", lit));

<a id="EXP-01"></a>
## EXP-01 · Literals hold any value, with their spelling and its problems where they were read from source

In [ ]:
assert(F.kindOf(lit)[0] === "literal" && form_equal(F.form(lit), new F.Form("literal", { value: 3n })) && E.DIALECT.validate(lit).length === 0);
assert(lit.patterns.literal.value.get() === 3n && lit.as_schema() === Term); // read through the readers, its dialect first
assert(F.dialectOf(lit) === "Expressions.Patterns.Expressions.Term" && F.dialectOf(lit.patterns) === F.dialectOf(lit));
for (const value of [0n, 2n ** 70n, -0, NaN, "", new Uint8Array(), false]) { // every basic scalar
  assert(E.DIALECT.validate(term(value)).length === 0 && F.same(term(value), term(E.literal(value))));
}
assert(typeof F.form(term(-0)).attributes.get("value") === "number" && Object.is(term(-0).patterns.literal.value.get(), -0));
assert(same(E.DIALECT.validate(term(E.literal(null))), ["a literal needs a value"]));
assert(same(E.DIALECT.validate(term((u: any) => u.literal((r: any) => r))), ["a literal needs a value"]));
assert(F.same(term((u: any) => u.literal((r: any) => r.value((p: any) => p.as_scalar("x")))), term("x"))); // the builders' own calls
// value is an unrestricted OfAny: a literal holds any value, a term value among them (CAP-233)
const held = term((u: any) => u.literal((r: any) => r.value(lit)));
assert(E.DIALECT.validate(held).length === 0 && F.same(F.form(held).attributes.get("value"), lit));
// a domain representation (PRI-023): its spelling, unvalidated, and a problem for each thing wrong with it
const spelled = term((u: any) => u.literal((r: any) => r.value((p: any) => p.as_scalar(16n)).text("0x10")));
assert(equal(F.form(spelled).attributes, map({ value: 16n, text: "0x10" })) && E.DIALECT.validate(spelled).length === 0);
assert(!F.same(spelled, term(16n))); // the spelling is part of the term
const invalid = term((u: any) => u.literal((r: any) => r.text("08").problems(["leading zeros", "not octal"]))); // no value
assert(same(E.DIALECT.validate(invalid), ["leading zeros", "not octal"]));
raises(ValueError, () => V.OfAny(invalid).value, "a literal needs a value");

<a id="EXP-02"></a>
## EXP-02 · Calls have a callable and ordered arguments

In [ ]:
let form = F.form(op);
assert(form.kind === "call" && equal(form.attributes, map({ callable: "f" })) && same(form.arguments.map((a: any) => a.patterns.literal.value.get()), [1n, "x", 3n]));
assert(F.same(form.arguments[2], lit) && op.patterns.call.callable.get() === "f");
assert(same(E.DIALECT.validate(op), ["'f' is not an operation Patterns defines"])); // written, and refused: Patterns defines no f
const nested = term((u: any) => u.call((c: any) => c.callable("not").arguments([E.DIALECT.spec(op)])));
assert(F.same(F.form(nested).arguments[0], op));
assert(same(E.DIALECT.validate(term((u: any) => u.call((c: any) => c))), ["a call needs a callable"]));
assert(same(E.DIALECT.validate(term((u: any) => u.call((c: any) => c.callable("")))), ["a call needs a callable"]));
assert(same(E.DIALECT.validate(term((u: any) => u.call((c: any) => c.callable(3n)))), ["a call's callable must be a str, got int"]));
const bad = term(E.operation("f", E.literal(null), (u: any) => u.call((c: any) => c.arguments([E.literal(null)]))));
assert(same(E.DIALECT.validate(bad), [
  "'f' is not an operation Patterns defines",
  "argument 0: a literal needs a value",
  "argument 1: a call needs a callable",
  "argument 1: argument 0: a literal needs a value",
]));

<a id="EXP-03"></a>
## EXP-03 · A term is a nested value: written into another it is copied, so it holds no cycle

In [ ]:
const pair = term(E.operation("g", op, op)); // each argument a copy of op
assert(F.form(pair).arguments.every((a) => F.same(a, op)));
assert([...F.walk(pair)].length === 1 + 2 * [...F.walk(op)].length); // nothing is shared: every term is visited
let again = term(pair);
assert(F.same(again, pair) && again !== pair);

<a id="EXP-04"></a>
## EXP-04 · Terms are built through mbse-schemas' builders, in a transaction: a term alone, or an expression's root

In [ ]:
const x = E.variable("x");
const source = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(E.operation("f", 1n))));
assert(Expression.ref && equal(F.form(F.term(source)).attributes, map({ callable: "f" })));
const [updated, copy] = store.transaction(() => [
  store.Expressions.Framework.Expression(source, (e: any) => e.root(E.operation("g", 2n))), // an update
  store.Expressions.Framework.Expression(source), // a clone
]);
assert(updated === source && equal(F.form(F.term(source)).attributes, map({ callable: "g" })));
assert(copy !== source && F.same(copy, source));
raises(ValueError, () => store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => e))), "no term is set");
assert(F.same(F.term(source.root), source)); // an expression, its root property, or the term itself
raises(TypeError, () => F.term(3n), "not a term: 3");
// A spec is a writer, a term value (copied), a native value (a literal) or a callable on the term value's builder.
assert(E.DIALECT.spec(x) instanceof E.Writer && E.DIALECT.spec(lit) instanceof F.Writer);
assert(E.DIALECT.spec(1n) instanceof E.Writer && E.DIALECT.spec((u: any) => u.variable((v: any) => v.name("x"))) instanceof F.Writer);
raises(TypeError, () => E.DIALECT.spec(BigInt), "expected a term, a writer or a native value, got <class 'int'>");
raises(TypeError, () => E.DIALECT.spec([1n]), "expected a term, a writer or a native value, got [1]");
raises(TypeError, () => F.spec(null), "expected a term, a writer or a native value, got None");
raises(TypeError, () => F.spec(1n, () => null), "expected a term, a writer or a native value, got 1"); // a dialect with no literal for it

<a id="EXP-05"></a>
## EXP-05 · The framework's schemas are an install: ordinary, registered schemas, which Patterns' install extends

In [ ]:
const before = JSON.ToJSON(store).Snapshot();
for (const install of INSTALLS) store.transaction(install);
assert(JSON.ToJSON(store).Snapshot() === before); // run again, each install writes nothing
let names = ["Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
  "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping", "Expressions.Patterns.Expressions.Quantifier",
  "Expressions.Framework.Error", "Expressions.Framework.Missing"];
const kinds = names.map((name) => store.registered(name));
const Pair: any = store.registered("Expressions.Framework.Pair");
const Union: any = store.registered("Expressions.Patterns.Expressions.Term"); // Patterns' union of its kinds
const schemas = [Term, Union, Expression, store.registered("Expressions.Framework.HasRoot"), Pair, ...kinds];
// the union of the dialects' `Term`s: each dialect's install adds its own, named after it; with Patterns the only dialect
// with terms (the standard dialects have none), Term is a union of one branch, which validates, as do the schemas holding it and the expressions (mbse-schemas 0.26.6)
assert(same(Term.branches.map((b: any) => b.name), ["patterns"]) && Term.branches[0].schema === Union);
assert(schemas.every((schema) => schema.validate().length === 0));
const lone = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(E.literal(1n))));
assert(Validators.Validate(store)(Expression, lone).length === 0);
assert(same(Union.branches.map((b: any) => b.name), ["literal", "call", "variable", "let", "list", "mapping", "quantifier",
  "error", "missing"]));
assert(same(Union.branches.map((b: any) => b.schema), kinds) && !kinds.some((kind) => kind.ref)); // value objects
assert(equal(kinds.map((k) => [...k.properties.keys()]), [["value", "text", "problems"], ["callable", "arguments"], ["name"],
  ["name", "value", "body"], ["elements"], ["entries"], ["name", "value", "body", "quantifier"], ["text", "terms"],
  ["kind"]]));
assert(kinds[4].properties.get("elements").schema.element === Term && kinds[5].properties.get("entries").schema.element === Pair);
assert(same([...Pair.properties.keys()], ["key", "value"]) && [...Pair.properties.values()].every((p: any) => p.schema === Term));
assert(kinds[1].properties.get("arguments").schema.element === Term && kinds[3].properties.get("body").schema === Term);
assert(kinds[6].properties.get("body").schema === Union); // a dialect's own kind holds its dialect's terms; a shared one, any
assert(kinds[7].properties.get("terms").schema.element === Term);
assert(Expression.ref && same([...Expression.properties.keys()], ["root"]) && Expression.properties.get("root").schema === Term);
const Dialect: any = store.registered("Expressions.Framework.Dialect"); // a kind union: each dialect adds its singleton's schema
assert(same(Dialect.branches.map((b: any) => b.name), ["isoiec11404", "ieee754", "ieee1164", "unicode", "patterns"]));
assert(Dialect.branches[4].schema === store.registered("Expressions.Patterns.Expressions.Dialect"));
assert(Dialect.branches[4].schema.ref && Dialect.branches[4].schema.singleton === "Expressions.Patterns.Expressions");
const HasRoot: any = store.registered("Expressions.Framework.HasRoot");
assert(same([...HasRoot.entry.links.keys()], ["schema", "expression"]) && HasRoot.uniques.length === 1
  && same([...HasRoot.uniques[0]], ["expression"]));
const OfExpression: any = store.registered("Schemas.OfExpression");
assert(OfExpression.adjacencies.get("expression").relation === HasRoot && OfExpression.adjacencies.get("expression").me === "schema");

<a id="EXP-06"></a>
## EXP-06 · Round trips through Plain, JSON and YAML: a term written nested, its kind its branch's name

In [ ]:
const expr = term(E.list_(E.operation("eq", 1n, 1.5), "s", new Uint8Array([1]), E.mapping([true, 1n], ["k", E.variable("v")])));
const expected = JSON.ToJSON(store)(Term, expr);
for (const [encode, decode] of [[Plain.ToPlain(store), Plain.FromPlain(store)], [JSON.ToJSON(store), JSON.FromJSON(store)],
  [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
  const back = decode(Term, encode(Term, expr));
  assert(back !== expr && F.same(back, expr) && E.DIALECT.validate(term(back), ["v"]).length === 0); // within a holder
  assert(JSON.ToJSON(store)(Term, back) === expected);
}
assert(JSON.ToJSON(store)(Term, term(E.operation("not", true))) // each term its dialect, then its kind
  === '{"patterns": {"call": {"callable": "not", "arguments": [{"patterns": {"literal": {"value": {"bool": true}}}}]}}}');
const alone = JSON.ToJSON(store)(Term, term(-0));
assert(alone === '{"patterns": {"literal": {"value": {"float": -0.0}}}}');
assert(Object.is((JSON.FromJSON(store)(Term, alone) as any).patterns.literal.value.get(), -0));
// A literal's value is an unrestricted OfAny's: written with its schema, one level (mbse-basis CAP-071).
for (const [value, wire] of [[1n, '{"int": 1}'], [1.5, '{"float": 1.5}'], [true, '{"bool": true}'], ["x", '{"str": "x"}'],
  [new Uint8Array([1]), '{"bytes": "AQ=="}'], [NaN, '{"float": "NaN"}']] as [unknown, string][]) {
  const written = JSON.ToJSON(store)(Term, term(value));
  assert(written === `{"patterns": {"literal": {"value": ${wire}}}}`);
  assert(F.same(JSON.FromJSON(store)(Term, written), term(value)));
}
// nothing set: only the branches are written
const all = ["literal", "call", "variable", "let", "list", "mapping", "error", "missing"];
assert(same(all.map((k) => JSON.ToJSON(store)(Term, term((u: any) => u[k]((r: any) => r)))),
  all.map((k) => `{"patterns": {"${k}": {}}}`)));
assert(JSON.ToJSON(store)(Term, term(E.mapping(["a", 1n])))
  === '{"patterns": {"mapping": {"entries": [{"key": {"patterns": {"literal": {"value": {"str": "a"}}}}, '
  + '"value": {"patterns": {"literal": {"value": {"int": 1}}}}}]}}}');
raises(Error, () => JSON.FromJSON(store)(Term, '{"operation": {}}'), "has no branch 'operation'"); // a dialect, then its kind
raises(Error, () => JSON.FromJSON(store)(Term, '{"patterns": {"operation": {}}}'), "patterns.operation: the union has no branch 'operation'");
// A store holding expressions, Patterns' dialect and its operations, and ISO/IEC 11404's and its domains (a packed one
// among them), round-trips whole, read into a store with the same installs: nothing lost, nothing written twice.
const defined = store.transaction(ISOIEC11404.define)[0];
store.transaction(() => {
  store.Expressions.Framework.Expression((e: any) => e.root(expr));
  const state = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
    (u: any) => u.isoiec11404((v: any) => v.state((e: any) => e.members(["IDLE", "RUN"]).ordered(false)))))));
  const mode = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union((u: any) => u.isoiec11404(
    (v: any) => v.packed((e: any) => e.codes([0n, 1n]).ordered(true)))))).packs((a: any) => a.insert(
    (e: any) => e.domain(state).representation(defined.get("octet")))));
  store.Expressions.ISOIEC11404.Expressions.Dialect(defined.get("dialect"), (b: any) => b.domains({
    state: (e: any) => e.domain(state), mode: (e: any) => e.domain(mode) }));
});
const snapshot = JSON.ToJSON(store).Snapshot();
const restored: any = new Builders.Store();
for (const install of INSTALLS) restored.transaction(install);
JSON.FromJSON(restored).Snapshot(snapshot);
assert(JSON.ToJSON(restored).Snapshot() === snapshot);
const restoredIso = restored.Instances.Expressions.ISOIEC11404.Expressions; // by its path
assert(same([...restoredIso.domains].map((e: any) => `${e.name}`).sort(), [
  "boolean", "integer", "octetstring", "octet", "modulo(256)", "state", "mode"].sort()));
assert(same([...FD.operations(restored.Instances.Expressions.Patterns.Expressions).keys()].sort(), [...Patterns.Domains.SIGNATURES.keys()].sort()));
const packed = restoredIso.domains({ name: "mode" }); // owned: found by its path, its dialect's then its name
assert(same(FD.kindOf(packed).slice(0, 2), ["isoiec11404", "packed"]) && equal(FD.properties(packed), map({ codes: [0n, 1n], ordered: true })));
assert(equal([...packed.packs].map((e: any) => [FD.nameOf(e.domain), FD.nameOf(e.representation)]), [["state", "octet"]])); // its links
assert(same(FD.kindOf(restoredIso.domains({ name: "state" })).slice(0, 2), ["isoiec11404", "state"]));

<a id="EXP-07"></a>
## EXP-07 · An expression is a reference object; a schema's expression is linked by `HasRoot`, an extension adjacency

In [ ]:
const [adult, schema] = store.transaction(() => {
  const adult = store.Expressions.Framework.Expression((e: any) => e.root(E.variable("person").age.ge(18n)));
  const schema = store.Schemas.OfExpression((s: any) => s.name("exp.Adult").result([store.Schemas.OfBool])
    .expression((a: any) => a.insert((e: any) => e.expression(adult))));
  return [adult, schema];
});
const linked = [...(new Builders.OfExpression.Builder(schema, false, store) as any).expression].map((entry: any) => entry.expression);
assert(linked.length === 1 && F.same(linked[0], adult));
assert(Validators.Validate(store)(Expression, adult).length === 0);
assert(V.OfAny(adult, { person: null }).value === null);
// Patterns' own holder refines the framework's Expression, its root narrowed to Patterns' Term: HasRoot links it, and
// it is read, validated and evaluated as any expression.
const Holder: any = store.registered("Expressions.Patterns.Expressions.Expression");
const [senior, elder] = store.transaction(() => {
  const senior = store.Expressions.Patterns.Expressions.Expression((e: any) => e.root(E.variable("person").age.ge(65n)));
  const elder = store.Schemas.OfExpression((s: any) => s.name("exp.Senior").result([store.Schemas.OfBool])
    .expression((a: any) => a.insert((e: any) => e.expression(senior))));
  return [senior, elder];
});
assert(Holder.refines === Expression && F.isExpression(Holder) && !F.isExpression(Term));
assert(same([...(new Builders.OfExpression.Builder(elder, false, store) as any).expression].map((e: any) => F.same(e.expression, senior)), [true]));
assert(Validators.Validate(store)(Holder, senior).length === 0 && E.DIALECT.validate(senior, ["person"]).length === 0);
assert(F.dialectOf(F.term(senior)) === E.DIALECT.union && V.OfAny(senior, { person: null }).value === null);

<a id="EXP-08"></a>
## EXP-08 · Validation and comparison work through the schemas

In [ ]:
assert(Validators.Validate(store)(Expression, adult).length === 0 && E.DIALECT.validate(adult, ["person"]).length === 0);

function compare(a: unknown, b: unknown) {
  return new C.OfUnion(Term, term(a)).compare(new C.OfUnion(Term, term(b)));
}

assert(compare(1n, 1n) === 0 && compare(1n, 2n) === null); // values are not ordered
assert(compare(1n, 1.0) === null); // another scalar
assert(compare(E.operation("f", 1n), E.operation("f", 1n)) === 0 && compare(E.operation("f", 1n), E.operation("f", 2n)) === null);
assert(compare(E.variable("x"), E.literal("x")) === null); // another branch

<a id="EXP-09"></a>
## EXP-09 · Terms read and write through the visitor protocols

In [ ]:
names = [...op.patterns.call.as_properties].map((p: any) => p.as_schema().name);
assert(same(names, ["callable", "arguments"])); // the properties set, schema unaware
const entries: [unknown, any][] = [];
op.patterns.call.arguments.as_value((a: any) => a.as_container((c: any) => c.as_entries((it: Iterator<[unknown, any]>) => {
  for (let next = it.next(); !next.done; next = it.next()) entries.push(next.value);
})));
assert(same(entries.map(([i]) => i), [0n, 1n, 2n]));
const values: unknown[] = [];
for (const [, element] of entries) {
  element.as_union((u: any) => values.push(u.patterns.literal.value.get())); // its dialect, then its kind
}
assert(same(values, [1n, "x", 3n]) && F.same(F.term(entries[0]![1]), term(1n)) && !F.isTerm(entries[0]![1])); // an element's reader holds a term
const seen: string[] = [];
term((u: any) => {
  seen.push(u.as_schema().constructor.name);
  return u.literal((r: any) => {
    seen.push(r.as_schema().name);
    return r;
  });
});
assert(same(seen, ["UnionData", "Expressions.Framework.Literal"])); // a dialect's spec is given its dialect's `Term`
// A writer writes through what it is given: a term value's union, a property, a container's element; where it is given
// a value of Term, it selects its dialect's branch first, as a callable written through the builders does.
const written = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root((p: any) => p.as_value((a: any) => a.as_union(
  (u: any) => u.patterns((d: any) => d.call((c: any) => c.callable("f").arguments([
    E.literal(1n), (y: any) => y.as_union((t: any) => t.patterns(E.variable("x").write))]))))))));
assert(F.same(written, term(E.operation("f", 1n, E.variable("x")))));

<a id="EXP-10"></a>
## EXP-10 · Variables and lets

In [ ]:
const v = term(E.variable("x"));
assert(F.nameOf(v) === "x" && F.kindOf(v)[0] === "variable");
assert(same(E.DIALECT.validate(v), ["variable 'x' is not bound"]) && E.DIALECT.validate(v, ["x"]).length === 0);
assert(same(E.DIALECT.validate(term((u: any) => u.variable((r: any) => r))), ["a variable needs a name"]));
assert(same(E.DIALECT.validate(term((u: any) => u.variable((r: any) => r.name(1n)))), ["a variable's name must be a str, got int"]));
const let_ = term(E.let_("x", 1n, E.variable("x")));
form = F.form(let_);
assert(equal(form.attributes, map({ name: "x" })) && F.same(form.arguments[0], term(1n)) && F.same(form.arguments[1], v));
assert(E.DIALECT.validate(let_).length === 0 && F.nameOf(let_) === "x");
assert(same(E.DIALECT.validate(term(E.let_("x", null, null))), ["a let needs a value", "a let needs a body"]));
assert(same(E.DIALECT.validate(term((u: any) => u.let((r: any) => r.value(E.literal(1n)).body(E.variable("x"))))), [
  "a let needs a name", "body: variable 'x' is not bound"]));
const sameName = term(E.let_("y", E.variable("y"), E.variable("y")));
assert(same(E.DIALECT.validate(sameName), ["value: variable 'y' is not bound"])); // the name is bound in the body only
const openLet = term(E.let_("x", null, null));
assert(same(F.form(openLet).arguments, [null, null]) && F.nameOf(term(E.literal(1n))) === null);
assert(F.same(term(E.DIALECT.make(F.form(openLet))), openLet));

<a id="EXP-11"></a>
## EXP-11 · Writers build terms with methods

In [ ]:
const person = E.variable("person");
const writer = person.age.ge(18n);
assert(writer instanceof E.Writer && typeof writer === "function");
form = F.form(term(writer));
assert(equal(form.attributes, map({ callable: "ge" })) && (form.arguments[1] as any).patterns.literal.value.get() === 18n);
const got = F.form(form.arguments[0]);
assert(equal(got.attributes, map({ callable: "get" })) && F.nameOf(got.arguments[0]) === "person"
  && (got.arguments[1] as any).patterns.literal.value.get() === "age");
assert((F.form(term(person.get("eq"))).arguments[1] as any).patterns.literal.value.get() === "eq"); // a property named like a method
const methods = [person.eq(1n), person.ne(1n), person.lt(1n), person.le(1n), person.gt(1n), person.ge(1n), person.and_(true), person.or_(true),
  person.not_(), person.implies(true), person.add(1n), person.sub(1n), person.mul(1n), person.neg(), person.has("x"),
  person.get("x")];
assert(same(methods.map((m) => F.form(term(m)).attributes.get("callable")), [
  "eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not", "implies", "add", "sub", "mul", "neg", "has", "get"]));
assert(term(E.literal(3n)).patterns.literal.value.get() === 3n && equal(F.form(term(E.operation("in", 1n, 2n))).attributes, map({ callable: "in" })));
const bound = term(E.let_("a", 1n, E.variable("a")));
assert(F.nameOf(bound) === "a" && F.same(term(bound), bound));
// JavaScript's own probes are not properties, nor are private names, as Python's are not
assert(person._private === undefined && person.then === undefined && person.toJSON === undefined && person[Symbol.iterator as any] === undefined);

<a id="EXP-13"></a>
## EXP-13 · Dialects, operations and domains are data: Patterns' singleton names its operations, and a standard dialect's owns its domains

In [ ]:
const definedAgain = store.transaction(Patterns.define)[0]; // Patterns' dialect's data; run again, it writes nothing
assert(store.transaction(Patterns.define)[0].get("dialect") === definedAgain.get("dialect"));
const basic = definedAgain.get("dialect");
assert(basic === store.Instances.Expressions.Patterns.Expressions && basic === store.singleton("Expressions.Patterns.Expressions")); // its path: its global name
const [Defines, Operates]: any[] = [store.registered("Expressions.Framework.Defines"), store.registered("Expressions.Framework.Operates")];
assert(same([...Defines.entry.links.keys()], ["dialect", "domain"]) && same([...Defines.entry.properties.keys()], ["name"]));
assert(equal(Defines.uniques.map((u: any) => [...u].sort()), [["dialect", "name"], ["domain"]]));
assert(same([...Operates.entry.links.keys()], ["dialect", "operation"]) && same([...Operates.entry.properties.keys()], ["name"]));
assert(equal(Operates.uniques.map((u: any) => [...u].sort()), [["dialect", "name"], ["operation"]]));
const DomainSchema: any = store.registered("Expressions.Framework.Domain");
assert(DomainSchema.ref && DomainSchema.adjacencies.get("dialect").owner && DomainSchema.adjacencies.get("dialect").relation === Defines);
// Its operations: expression schemas its module defines, named by the dialect; their variables, in order, its arguments.
const operations = new Map<string, any>([...basic.operations].map((entry: any) => [`${entry.name}`, entry.operation]));
assert(equal(FD.operations(basic), operations)); // what validation reads, from the store of the term's holder
assert(same([...operations.keys()], [...Patterns.Domains.SIGNATURES.keys()])); // infer's vocabulary: the same names, in order
const arities = new Map([...operations].map(([name, o]) => [name, FD.arity(o)])); // each its pattern's variables
assert(equal(arities, new Map([...operations].map(([name, o]) => [name, o.pattern.variables.size]))));
assert(equal(arities, new Map([...Patterns.Domains.SIGNATURES].map(([name, s]) => [name, s.arity()]))));
assert([...operations].every(([name, o]) => o === store.registered(`Expressions.Patterns.Expressions.${name}`) && o.validate().length === 0));
const [add, number] = [operations.get("add"), store.registered("Expressions.Patterns.Expressions.Number")];
assert(same([...add.pattern.variables.keys()], ["a", "b"]) && add.pattern.variables.get("a").schema === number && add.result === number);
assert(same([...operations.get("shl").pattern.variables.keys()], ["value", "count"])); // the order declared: arguments are positional
assert(same([...operations.get("in").pattern.variables.keys()], ["value", "collection"]));
assert(operations.get("in").pattern.variables.get("collection").schema === store.registered("Expressions.Patterns.Expressions.Collection"));
assert(same(operations.get("eq").result.schemas, [store.registered("Schemas.OfBool")])); // a result is an OfAny
assert(operations.get("get").result.schemas.length === 0 && operations.get("not").pattern.variables.get("a").schema === store.registered("Schemas.OfBool"));
const reread: any = new Builders.Store(); // the order survives a snapshot
JSON.FromJSON(reread).Snapshot(JSON.ToJSON(store).Snapshot());
assert(same([...reread.registered("Expressions.Patterns.Expressions.shl").pattern.variables.keys()], ["value", "count"]));
assert(same(E.DIALECT.validate(term(E.operation("contains", 1n, 2n))), ["'contains' is not an operation Patterns defines"]));
assert(same(E.DIALECT.validate(term(E.operation("add", 1n))), ["add takes 2 arguments, got 1"]));
assert(!operations.has("convert") && !operations.has("pack")); // deferred with domain inference
// a term alone, in no holder, belongs to no store: its calls cannot be validated
const unheld = store.transaction(() => store.Expressions.Framework.Term(E.DIALECT.spec(E.operation("add", 1n, 2n))));
raises(TypeError, () => E.DIALECT.validate(unheld), "the value does not belong to a store");
assert(E.DIALECT.validate(term(unheld)).length === 0); // written into a holder, it validates
const bare: any = new Builders.Store(); // installed, but its operations not defined: the data names none, and every call is refused
for (const install of INSTALLS) bare.transaction(install);
const unnamed = bare.transaction(() => F.term(bare.Expressions.Framework.Expression((e: any) => e.root(E.operation("add", 1n, 2n)))));
assert(same(E.DIALECT.validate(unnamed), ["'add' is not an operation Patterns defines"]));
assert(basic.operations({ name: "add" }) === operations.get("add")); // by its path, its dialect's then its name
// Its domains, none: Patterns' values take the standard dialects' (ISO/IEC 11404's here), owned by name, each related
// to its schema, an OfAny of its scalar, both ways.
assert([...basic.domains].length === 0);
const iso = store.Instances.Expressions.ISOIEC11404.Expressions;
const domains = new Map<string, any>([...iso.domains].map((entry: any) => [`${entry.name}`, entry.domain]));
assert(same([...domains.keys()], ["boolean", "integer", "octetstring", "octet", "modulo(256)", "state", "mode"])); // EXP-06's too
assert(domains.get("octet") === store.transaction(ISOIEC11404.define)[0].get("octet") && domains.get("octet") === iso.domains({ name: "octet" }));
// `octet` holds codes, with no arithmetic; `modulo(256)` wraps its arithmetic, its schema spelled as a name can be.
const [octet, wrap8] = ["octet", "modulo_256"].map((name) => store.registered(`Expressions.ISOIEC11404.Expressions.${name}`));
assert(same(octet.schemas, [store.registered("Schemas.OfInt")]) && same(wrap8.schemas, octet.schemas));
assert(same([...domains.get("octet").schema].map((e: any) => e.schema), [octet])
  && same([...domains.get("modulo(256)").schema].map((e: any) => e.schema), [wrap8]));
assert(equal([...new Builders.OfAny.Builder(wrap8, false, store).domain].map((e: any) => [FD.kindOf(e.domain).slice(0, 2),
  FD.properties(e.domain).get("modulus")]), [[["isoiec11404", "modulo"], 256n]])); // its schema's
assert(same(FD.kindOf(domains.get("octet")).slice(0, 2), ["isoiec11404", "octet"]) && FD.nameOf(domains.get("octet")) === "octet");
assert(JSON.ToJSON(store)(DomainSchema, domains.get("modulo(256)")).split('"modulus"').length === 2); // a document of its own, by its path